# Task 3 (D3 Phase A) — Answerer: citations, refusal & the strategy comparison

**Owner: WAFIQ.** Deliverable artifact for the zero-shot answerer. The *logic* lives in
`src/csai415/answer.py` (importable + tested); this notebook only **runs** it and renders the
evidence:

1. one grounded answer end-to-end (smoke),
2. the citation-strategy comparison — **numbered vs hybrid vs posthoc** × **Qwen-3B vs Groq-70B ceiling**,
   scored on a faithfulness proxy × p95 latency.

Backend + strategy are env-driven (`CSAI415_ANSWERER`, `CSAI415_CITE_MODE`) — no code changes to swap.
Develop against **Groq** (free, fast, no GPU); produce the graded **Qwen-3B** numbers on the Colab GPU.

## 0 · Environment (Colab clones the repo; local is a no-op)

In [ ]:
import sys, subprocess, os
from pathlib import Path

REPO_URL = "https://github.com/waf-iq/special-topics.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not Path("special-topics").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("special-topics")
    subprocess.run(["git", "pull", "--ff-only"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
else:
    # local: assume the notebook runs from repo root or notebooks/ — make src importable
    root = Path.cwd()
    if (root / "notebooks").exists() is False and (root.parent / "src").exists():
        os.chdir(root.parent)
    sys.path.insert(0, str(Path.cwd() / "src"))

print("cwd:", Path.cwd())

In [ ]:
# Backend config. For dev: Groq (free, instant, no GPU). For graded Qwen numbers on the
# Colab GPU: start Ollama + `ollama pull qwen2.5:3b-instruct`, then set CSAI415_ANSWERER to it.
from getpass import getpass

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("GROQ_API_KEY (free, console.groq.com): ")

os.environ["CSAI415_ANSWERER"] = "groq:llama-3.3-70b-versatile"  # dev backend
os.environ["CSAI415_CITE_MODE"] = "numbered"

from csai415 import answer as ans
print("backend:", ans.current_backend())

## 1 · Smoke: one grounded answer

`generate_answer(query, citations, contexts)` with `contexts[i]` aligned to `citations[i]`.
Here we pass a tiny hand-made context list; swap in the real executor output once Task 1/2/7
land (`GraphRAGExecutor.answer(...)` returns aligned `.citations` / `.contexts`).

In [ ]:
from csai415.graphrag import Citation

# fixture (query, contexts) — replace with real retrieved chunks via the executor
query = "What problem does the attention mechanism solve in sequence models?"
contexts = [
    "Attention lets a model weigh all input positions when producing each output token, "
    "removing the fixed-length bottleneck of RNN encoder-decoders on long sequences.",
    "BERT is a bidirectional encoder pre-trained with masked language modeling.",
    "Stochastic gradient descent updates parameters using mini-batches.",
]
citations = [Citation(f"arxiv:p{i}:0", f"p{i}", f"Paper {i}", "1-2", 1.0) for i in range(len(contexts))]

text, used = ans.generate_answer(query, citations, contexts)
print("ANSWER:\n", text)
print("\nCITED:", used, "->", [citations[i - 1].chunk_id for i in used])

## 2 · Strategy × backend comparison (the deep-dive table)

Sweep `CSAI415_CITE_MODE` × `CSAI415_ANSWERER`. `faith_proxy` = lexical overlap of the answer
with the cited contexts (a cheap stand-in until Ahmad's RAGAS harness lands — replace the
`faith_proxy` column with the real RAGAS faithfulness row at integration). Latency is wall-clock
per call; run a few questions for a meaningful p95. Missing backends (e.g. no local Ollama) are
skipped, not fatal.

In [ ]:
import re, time
import numpy as np, pandas as pd

def faith_proxy(answer: str, ctxs) -> float:
    aw = set(re.findall(r"[a-z]{3,}", (answer or "").lower()))
    bw = set(re.findall(r"[a-z]{3,}", " ".join(ctxs).lower()))
    return len(aw & bw) / len(aw) if aw else 0.0

# (label, CSAI415_ANSWERER). Add the tuned model once D4 lands: ("qwen-tuned", "qwen2.5-3b-csai415").
BACKENDS = [
    ("groq-70b", "groq:llama-3.3-70b-versatile"),
    ("qwen-3b", "qwen2.5:3b-instruct"),
]
STRATEGIES = ["numbered", "hybrid", "posthoc"]
QUESTIONS = [(query, citations, contexts)]  # TODO(WAFIQ): expand to ~5-10 example questions

rows = []
for blabel, bspec in BACKENDS:
    os.environ["CSAI415_ANSWERER"] = bspec
    for strat in STRATEGIES:
        os.environ["CSAI415_CITE_MODE"] = strat
        lats, faiths, ncite = [], [], []
        try:
            for q, cits, ctxs in QUESTIONS:
                t0 = time.perf_counter()
                a, u = ans.generate_answer(q, cits, ctxs)
                lats.append((time.perf_counter() - t0) * 1000)
                faiths.append(faith_proxy(a, [ctxs[i - 1] for i in u] or ctxs))
                ncite.append(len(u))
        except Exception as e:
            rows.append({"backend": blabel, "strategy": strat, "status": f"skip: {type(e).__name__}"})
            continue
        rows.append({
            "backend": blabel, "strategy": strat, "status": "ok",
            "faith_proxy": round(float(np.mean(faiths)), 3),
            "avg_citations": round(float(np.mean(ncite)), 2),
            "p95_latency_ms": round(float(np.percentile(lats, 95)), 1),
        })

pd.DataFrame(rows)

## 3 · Verdict

_Fill in after running on the Colab GPU with the real Qwen-3B + ~5–10 questions:_

- Winning strategy and why (faithfulness vs p95 trade-off).
- Qwen-3B vs Groq-70B ceiling gap.
- Whether the 2s p95 target holds at the chosen `max_tokens` / context truncation.
- AI log: approaches compared + share link (per the deliverable checklist).